# Module 5 — Ingestion de documents

## Objectif
Charger `data/support_kb/*.md`, les **découper** et les indexer dans Qdrant avec un **ID déterministe**.

### Analogie
Le **chunk overlap** = la marge entre deux photocopies : on évite de couper une phrase en deux.

### Tips
> ID = hash du contenu → ré-ingérer ne crée pas de doublon (idempotence).

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070, SKEXP0001
using Microsoft.Extensions.AI;            // IEmbeddingGenerator (moderne)
using Microsoft.Extensions.VectorData;    // attributs VectorStore + collection
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.Qdrant;
using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint = new Uri("http://localhost:11434");
// Embeddings MODERNES : IEmbeddingGenerator (remplace ITextEmbeddingGenerationService)
var embed = Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text", endpoint)
    .Build().GetRequiredService<IEmbeddingGenerator<string, Embedding<float>>>();
var store = new QdrantVectorStore(new QdrantClient("localhost"), ownsClient: true);
ulong Id(string s) => BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)), 0);

In [ ]:
// Un enregistrement Qdrant : clé + texte + vecteur (768 = dimension de nomic-embed-text)
public sealed class Doc {
    [VectorStoreKey] public ulong Id { get; set; }
    [VectorStoreData] public string Text { get; set; } = "";
    [VectorStoreData(IsIndexed = true)] public string Source { get; set; } = "";
    [VectorStoreVector(768, DistanceFunction = DistanceFunction.CosineSimilarity)]
    public ReadOnlyMemory<float> Vector { get; set; }
}

In [ ]:
var collection = store.GetCollection<ulong, Doc>("support_kb");
await collection.EnsureCollectionExistsAsync();

IEnumerable<string> Chunk(string t, int max = 600) =>
    t.Split("\n\n", StringSplitOptions.RemoveEmptyEntries)
     .Select(p => p.Length <= max ? p.Trim() : p.Substring(0, max).Trim())
     .Where(p => p.Length > 0);

var dir = Path.Combine("..", "data", "support_kb");
int n = 0;
foreach (var file in Directory.GetFiles(dir, "*.md")) {
    var source = Path.GetFileNameWithoutExtension(file);
    foreach (var chunk in Chunk(File.ReadAllText(file))) {
        var v = (await embed.GenerateAsync(chunk)).Vector;
        await collection.UpsertAsync(new Doc { Id = Id(chunk), Text = chunk, Source = source, Vector = v });
        n++;
    }
}
Console.WriteLine($"{n} chunks indexés depuis {dir}");

**À observer** : relance la cellule → pas de doublon (ID = hash, `UpsertAsync`).

## Récap
- Charger → découper → ID déterministe → indexer, avec la **source** en métadonnée.